In [0]:
from pyspark.sql.types import StringType
import pyspark.sql.functions as F

In [0]:
df = spark.read.table("workspace.bronze.erp_px_cat_g1v2")
display(df)

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))
        

In [0]:
df.printSchema()

In [0]:
df.select([F.sum(F.col(c).isNull().cast("int")) for c in df.columns]).show()

In [0]:
df.select(F.length(F.col("ID"))).distinct().show()

In [0]:
df.select(F.col("CAT")).distinct().show()
df.select(F.col("SUBCAT")).distinct().show()
df.select(F.col("MAINTENANCE")).distinct().show()

In [0]:
df = df.withColumn(
    "MAINTENANCE",
    F.when(F.upper(F.col("MAINTENANCE")) == "YES", F.lit(True))
     .when(F.upper(F.col("MAINTENANCE")) == "NO", F.lit(False))
     .otherwise(None)
)

In [0]:
RENAME_MAP ={
    "ID": "id",
    "CAT": "category",
    "SUBCAT": "subcategory",
    "MAINTENANCE": "maintenance"
}

df =df.withColumnsRenamed(RENAME_MAP)


In [0]:
display(df)

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.erp_product_categories")

In [0]:
%sql
SELECT *
FROM workspace.silver.erp_product_categories